# 7a단계: 경기장 선에 야드 값 붙이기 준비

**목표:** 선수 위치를 "그려진 선 기준 야드"로 바꾸기 위한 첫 단계. 실제 거리(m)는 쓰지 않아요.
선마다 야드 값을 붙이고, 선과 선 사이는 비율로 계산해요 (예: 10야드 선과 20야드 선 사이 60% 지점 = 16야드).

이 노트북은 참조 화면 4장에서 흰 선을 자동으로 찾아 **번호(H1, H2… / V1, V2…)**를 붙인 그림을 만들어요.
그림을 보고 선마다 야드 값을 정해 알려주시면, 다음 노트북(7b)에서 매 프레임 선수 위치를 야드로 바꿔요.

- **H** = 화면에서 가로에 가까운 선 (위 → 아래 순서), **V** = 세로에 가까운 선 (왼쪽 → 오른쪽 순서)
- 선 찾기: 흰색 HSV 마스크(2단원) → Hough 직선 조각 → 같은 직선끼리 묶기
- 못 찾은 선이 있으면 격자 좌표로 직접 두 점을 적을 수 있어요

**런타임: CPU로 충분** (1~2분)

In [ ]:
import os, json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font); plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

FOLDER_PATH = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
EVAL_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs", "eval")
LINE_DIR = os.path.join(EVAL_DIR, "field_lines"); os.makedirs(LINE_DIR, exist_ok=True)

# 참조 화면 4장 (run_pipeline의 참조와 같음): (프레임, 경기장 ROI)
KNOWN = {
    "DJI_20250913133612_0144_D": (413, [[245, 530], [1860, 543], [1919, 776], [1919, 1079], [107, 1079]]),
    "DJI_20250913135020_0156_D": (167, [[0, 0], [1632, 0], [1712, 810], [0, 787]]),
    "DJI_20250913133530_0143_D": (1, [[270, 459], [1880, 469], [1919, 612], [1919, 1079], [119, 1079]]),
    "DJI_20250913135111_0157_D": (1, [[0, 0], [1675, 0], [1761, 737], [0, 733]]),
}


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")


def read_frame(stem, fid):
    """처음부터 순서대로 읽어 fid번째(1부터) 프레임 (v3와 같은 프레임 번호)"""
    cap = cv2.VideoCapture(video_path_of(stem)); img = None
    for _ in range(fid):
        ok, img = cap.read()
        assert ok, f"{stem}: {fid}프레임을 읽을 수 없습니다"
    cap.release()
    return img

In [ ]:
# ① 흰 선 찾기 함수
def white_mask(img, roi):
    """경기장(ROI) 안의 흰 선 마스크 (2단원: HSV 마스크 + 형태학)"""
    H, W = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    m = cv2.inRange(hsv, (0, 0, 170), (180, 60, 255))
    field = np.zeros((H, W), np.uint8)
    cv2.fillPoly(field, [np.asarray(roi, np.int32).reshape(-1, 1, 2)], 255)
    field = cv2.dilate(field, np.ones((41, 41), np.uint8))           # 경계선 자체도 포함되게 조금 넓힘
    return cv2.morphologyEx(m & field, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))


def find_lines(img, roi, min_total=250):
    """흰 선 조각(Hough) → 같은 직선끼리 묶기 → [{'p1','p2','len','dir'}] (dir: 'H' 가로에 가까움 / 'V' 세로에 가까움)"""
    m = white_mask(img, roi)
    segs = cv2.HoughLinesP(m, 1, np.pi / 360, 80, minLineLength=80, maxLineGap=30)
    if segs is None:
        return []
    segs = segs.reshape(-1, 4).astype(float)
    items = []
    for x1, y1, x2, y2 in segs:
        th = np.arctan2(y2 - y1, x2 - x1) % np.pi                      # 0~π
        nx, ny = -np.sin(th), np.cos(th)
        items.append({"th": th, "rho": ((x1 + x2) / 2) * nx + ((y1 + y2) / 2) * ny, "pts": [(x1, y1), (x2, y2)],
                      "len": np.hypot(x2 - x1, y2 - y1)})
    items.sort(key=lambda d: -d["len"])
    groups = []
    for it in items:
        for g in groups:
            dth = abs(it["th"] - g["th"]); dth = min(dth, np.pi - dth)
            if dth < np.deg2rad(3):
                nx, ny = -np.sin(g["th"]), np.cos(g["th"])
                r = np.mean([px * nx + py * ny for px, py in it["pts"]])
                if abs(r - g["rho"]) < 12:
                    g["pts"] += it["pts"]; g["len"] += it["len"]; break
        else:
            groups.append({"th": it["th"], "rho": it["rho"], "pts": list(it["pts"]), "len": it["len"]})
    out = []
    for g in groups:
        if g["len"] < min_total:
            continue
        P = np.array(g["pts"])
        vx, vy, x0, y0 = cv2.fitLine(P.astype(np.float32), cv2.DIST_L2, 0, 0.01, 0.01).ravel()
        t = (P[:, 0] - x0) * vx + (P[:, 1] - y0) * vy
        p1 = (x0 + vx * t.min(), y0 + vy * t.min()); p2 = (x0 + vx * t.max(), y0 + vy * t.max())
        out.append({"p1": p1, "p2": p2, "len": float(t.max() - t.min()), "dir": "H" if abs(vx) > abs(vy) else "V"})
    # 번호: 가로 선은 위 → 아래, 세로 선은 왼쪽 → 오른쪽
    hs = sorted([l for l in out if l["dir"] == "H"], key=lambda l: (l["p1"][1] + l["p2"][1]) / 2)
    vs = sorted([l for l in out if l["dir"] == "V"], key=lambda l: (l["p1"][0] + l["p2"][0]) / 2)
    for i, l in enumerate(hs, 1):
        l["name"] = f"H{i}"
    for i, l in enumerate(vs, 1):
        l["name"] = f"V{i}"
    return hs + vs


def draw_lines(img, lines, roi=None, grid=True):
    vis = img.copy()
    H, W = vis.shape[:2]
    if grid:
        for x in range(0, W, 100):
            cv2.line(vis, (x, 0), (x, H), (200, 200, 200), 1)
            cv2.putText(vis, str(x), (x + 2, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        for y in range(0, H, 100):
            cv2.line(vis, (0, y), (W, y), (200, 200, 200), 1)
            cv2.putText(vis, str(y), (2, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
    for l in lines:
        col = (0, 0, 255) if l["dir"] == "H" else (255, 0, 255)
        p1 = tuple(int(v) for v in l["p1"]); p2 = tuple(int(v) for v in l["p2"])
        cv2.line(vis, p1, p2, col, 3)
        t = 0.5 if l["dir"] == "H" else 0.35
        mx, my = int(p1[0] + (p2[0] - p1[0]) * t), int(p1[1] + (p2[1] - p1[1]) * t)
        for th, cc in ((7, (0, 0, 0)), (2, (255, 255, 255))):
            cv2.putText(vis, l["name"], (mx + 6, my - 8), cv2.FONT_HERSHEY_SIMPLEX, 1.1, cc, th)
    return vis

In [ ]:
# ② 참조 화면 4장: 선 찾기 → 번호 그림 표시 + Drive에 저장 (다운로드해서 표시해도 돼요)
DETECTED = {}
for stem, (fid, roi) in KNOWN.items():
    img = read_frame(stem, fid)
    lines = find_lines(img, roi)
    DETECTED[stem] = {"frame": fid, "lines": {l["name"]: {"p1": [round(float(v), 1) for v in l["p1"]], "p2": [round(float(v), 1) for v in l["p2"]],
                                                          "len": round(l["len"])} for l in lines}}
    vis = draw_lines(img, lines)
    cv2.imwrite(os.path.join(LINE_DIR, f"{stem[-6:-2]}_lines.png"), vis)
    cv2.imwrite(os.path.join(LINE_DIR, f"{stem[-6:-2]}_clean.png"), img)              # 선 없는 원본 (직접 그려 표시할 때)
    plt.figure(figsize=(20, 11.3)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"{stem[-6:-2]} · f{fid} · 빨강 H = 가로 방향 선, 보라 V = 세로 방향 선 · 격자 100px", fontsize=15)
    plt.tight_layout(); plt.show()
    print(f"{stem[-6:-2]}: 가로 {sum(k[0] == 'H' for k in DETECTED[stem]['lines'])}개 · 세로 {sum(k[0] == 'V' for k in DETECTED[stem]['lines'])}개")
json.dump(DETECTED, open(os.path.join(LINE_DIR, "lines_detected.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print(f"\n✅ 그림 · 좌표 저장 → {LINE_DIR}")

## ③ 알려주실 것

그림마다 선 번호에 아래처럼 값을 붙여 주세요. 형식은 자유롭게 말로 알려주셔도 돼요.

| 값 | 뜻 |
|---|---|
| 숫자 (예: `0`, `10`, `20`) | 그 선이 몇 야드 선인지. 엔드존 라인 = 0 같은 식으로 정하신 기준 그대로 |
| `"사이드1"` / `"사이드2"` | 폭 방향 경계 (화면에서 위쪽/왼쪽 쪽을 사이드1로) |
| `"엔드1"` / `"엔드2"` | 엔드라인(엔드존 뒤 끝 선)이 따로 있으면 |
| `"무시"` | 야드 계산과 상관없는 선 (축구 박스 등) |

**예시** (번호와 값은 가짜예요):
```
0144: H1 = 사이드1, H4 = 사이드2, V3 = 30, V4 = 40, V1 = 무시, V2 = 무시
0144 못 찾은 선: (230, 1070)-(270, 530) = 20야드
```

- 그림에 없는 선은 격자 좌표로 두 점을 적어 주세요 (못 찾은 선)
- 엔드존과 야드 방향(어느 쪽이 0야드인지)도 같이 알려주세요
- 그림 파일은 Drive `pipeline_outputs/eval/field_lines/`에 있어요. `_clean.png`에 직접 그려서 보내주셔도 돼요